# EasyDE V2 — Exploratory Results Dashboard

This notebook reads all pipeline results across contrasts and strata, and produces:

1. **DE feature counts** — Boxplot of final-model DEG counts per contrast × cell type (RUVseq or Base fallback)
2. **FGSEA heatmaps** — Top significant pathways per contrast × cell type
3. **Diagonal pathway heatmap** — Pathways clustered along the diagonal to reveal contrast/cell-type specificity
4. **Pipeline status overview** — Heatmap of completion status across all contrasts × strata
5. **Export** — Save final DESeq2 results per stratum, with Ensembl ID mapping and PankBase manifest

In [ ]:
# ===========================================================================
# Section 0 — Setup
# ===========================================================================
suppressPackageStartupMessages({
  library(tidyverse)
  library(data.table)
  library(pheatmap)
  library(scales)
  library(RColorBrewer)
})

# --- Project root (adjust if needed) ---
PROJECT_ROOT <- normalizePath(file.path(getwd(), ".."), mustWork = TRUE)
RESULTS_DIR  <- file.path(PROJECT_ROOT, "results")

# --- Output directory for exported final results (set your path here) ---
OUT_DIR <- ""   # e.g. "/nfs/lab/lbrusman/pankbase/DE_results_EasyDE/final_results"

# --- File suffix for exported DESeq2 results ---
RESULTS_SUFFIX <- "DESEQ2"

cat("Project root:", PROJECT_ROOT, "\n")
cat("Results dir: ", RESULTS_DIR, "\n")
if (nzchar(OUT_DIR)) cat("Output dir:  ", OUT_DIR, "\n") else cat("Output dir:   (not set — set OUT_DIR to enable export)\n")

## Section 1 — Load All Contrast Summaries

In [ ]:
# Discover all contrast directories that have a summary/contrast_summary.csv
summary_files <- Sys.glob(file.path(RESULTS_DIR, "*", "summary", "contrast_summary.csv"))
cat(sprintf("Found %d contrast summary files:\n", length(summary_files)))
for (f in summary_files) cat("  ", basename(dirname(dirname(f))), "\n")

# Read and bind all summaries
deseq_all <- lapply(summary_files, function(f) {
  df <- read.csv(f, stringsAsFactors = FALSE)
  df
}) %>% bind_rows()

cat(sprintf("\nTotal rows: %d  |  Contrasts: %d  |  Strata: %d\n",
    nrow(deseq_all),
    n_distinct(deseq_all$contrast),
    n_distinct(deseq_all$biosample)))

# --- Load final (best-model) DESeq2 results across all contrasts ---
final_files <- Sys.glob(file.path(RESULTS_DIR, "*", "summary", "*_deseq_final.tsv"))
cat(sprintf("\nFound %d final DESeq2 result files\n", length(final_files)))

deseq_final <- lapply(final_files, function(f) {
  contrast_id <- basename(dirname(dirname(f)))
  df <- fread(f, sep = "\t", header = TRUE, stringsAsFactors = FALSE)
  if ("skipped" %in% colnames(df)) return(NULL)
  if (!"contrast" %in% colnames(df)) df$contrast <- contrast_id
  df
}) %>%
  Filter(Negate(is.null), .) %>%
  bind_rows()

if (nrow(deseq_final) > 0) {
  cat(sprintf("Final results: %d genes across %d contrasts, %d strata\n",
      nrow(deseq_final), n_distinct(deseq_final$contrast), n_distinct(deseq_final$stratum)))
  cat(sprintf("  Model breakdown: %s\n",
      paste(names(table(deseq_final$model)), table(deseq_final$model),
            sep = "=", collapse = ", ")))
} else {
  cat("No final DESeq2 files found. Run the pipeline with updated 07_aggregate_results.R first.\n")
}

# Quick preview
deseq_all %>%
  dplyr::select(biosample, contrast, status, n_degs_base, n_degs_ruv,
                n_pathways_base, n_pathways_ruv) %>%
  head(20)

## Section 2 — DE Features Boxplot (Final Model: RUVseq or Base fallback)

Uses the `_deseq_final.tsv` files which contain RUVseq results where available
and fall back to Base DESeq2 for `success_base_only` strata. Points are colored
by cell type, boxplots are outlined by model (solid = RUVseq, dashed = Base).

In [ ]:
# Define contrast display order
contrast_order <- c(
  "Age", "Gender", "BMI",
  "Diabetic_vs_ND", "Prediabetic_vs_ND", "Prediabetic_vs_Diabetic",
  "ND_vs_T2D", "ND_vs_T1D",
  "HbA1c", "C_peptide",
  "AAB_ZNT8", "AAB_IAA", "AAB_IA2"
)

# --- Summarise DEG counts from the final best-model file ---
deg_summary <- deseq_final %>%
  group_by(stratum, contrast, model, formula) %>%
  summarise(
    n_degs = sum(!is.na(padj) & padj < 0.05 & abs(log2FoldChange) > 0, na.rm = TRUE),
    n_genes = n(),
    .groups = "drop"
  ) %>%
  rename(biosample = stratum)

df_plot <- deg_summary %>%
  filter(contrast %in% contrast_order) %>%
  mutate(contrast = factor(contrast, levels = contrast_order))

cat(sprintf("Plotting %d contrast x strata combinations (%d RUVseq, %d Base)\n",
    nrow(df_plot),
    sum(df_plot$model == "RUVseq"),
    sum(df_plot$model == "Base")))

options(repr.plot.width = 14, repr.plot.height = 7)

ggplot(df_plot, aes(x = contrast, y = n_degs)) +
  geom_boxplot(aes(group = contrast), fill = "grey90", color = "grey50",
               outlier.shape = NA) +
  geom_point(aes(color = biosample, shape = model),
             position = position_jitter(width = 0.18, height = 0),
             size = 2.6, alpha = 0.9) +
  scale_shape_manual(values = c("RUVseq" = 16, "Base" = 17),   # circle vs triangle
                     name = "Model") +
  scale_y_continuous(
    trans = scales::pseudo_log_trans(sigma = 1),
    breaks = c(0, 1, 10, 100, 1000, 10000),
    labels = scales::comma,
    expand = expansion(mult = c(0, 0.05))
  ) +
  labs(
    x = NULL,
    y = "Number of DE features (final model)",
    color = "Cell type",
    title = "EasyDE V2 — DE features per contrast (RUVseq where available, Base fallback)",
    caption = expression(circle ~ "= RUVseq" ~~ triangle ~ "= Base (paired/no RUV)")
  ) +
  theme_classic(base_size = 12) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "right"
  )

## Section 3 — Load All FGSEA Results

In [ ]:
# Read merged fGSEA results from each contrast's summary folder
fgsea_ruv_files  <- Sys.glob(file.path(RESULTS_DIR, "*", "summary", "*_fgsea_ruv.tsv"))
fgsea_base_files <- Sys.glob(file.path(RESULTS_DIR, "*", "summary", "*_fgsea_base.tsv"))

read_fgsea <- function(path) {
  contrast_id <- basename(dirname(dirname(path)))
  df <- fread(path, sep = "\t", header = TRUE, stringsAsFactors = FALSE)
  if (nrow(df) == 0) return(NULL)
  # Check for skip sentinels
  if ("skipped" %in% colnames(df)) return(NULL)
  df$contrast <- contrast_id
  df
}

fgsea_ruv <- lapply(fgsea_ruv_files, read_fgsea) %>%
  Filter(Negate(is.null), .) %>%
  bind_rows()

fgsea_base <- lapply(fgsea_base_files, read_fgsea) %>%
  Filter(Negate(is.null), .) %>%
  bind_rows()

cat(sprintf("fGSEA-RUV:  %d rows across %d contrasts, %d strata\n",
    nrow(fgsea_ruv), n_distinct(fgsea_ruv$contrast), n_distinct(fgsea_ruv$stratum)))
cat(sprintf("fGSEA-base: %d rows across %d contrasts, %d strata\n",
    nrow(fgsea_base), n_distinct(fgsea_base$contrast), n_distinct(fgsea_base$stratum)))

## Section 4 — FGSEA Heatmap: Top Pathways per Contrast × Cell Type

Filter for highly significant pathways, select top N per group using a composite score,
then display NES values as a heatmap.

In [ ]:
# --- Use RUV fGSEA results ---
gsea <- fgsea_ruv %>%
  filter(!is.na(padj), !is.na(NES), size >= 5)

# Keep pathways that are significant (padj < 0.05) in at least one stratum×contrast
sig_pathways <- gsea %>%
  group_by(pathway) %>%
  filter(min(padj) < 0.01) %>%
  ungroup()

# Score: -log10(padj) * |NES|, then pick top 5 per stratum×contrast
top_pathways <- sig_pathways %>%
  group_by(pathway) %>%
  mutate(nes_sd = sd(NES, na.rm = TRUE)) %>%
  ungroup() %>%
  filter(nes_sd >= 0.5) %>%
  group_by(stratum, contrast) %>%
  mutate(score = -log10(pmax(padj, 1e-300)) * abs(NES)) %>%
  slice_max(order_by = score, n = 5) %>%
  ungroup()

selected_paths <- unique(top_pathways$pathway)
cat(sprintf("Selected %d pathways with high variability and significance\n", length(selected_paths)))

In [ ]:
# Build NES matrix: pathway × (celltype:contrast)
heatmap_data <- sig_pathways %>%
  filter(pathway %in% selected_paths) %>%
  mutate(col_label = paste0(stratum, ":", contrast)) %>%
  reshape2::dcast(pathway ~ col_label, value.var = "NES") %>%
  tibble::column_to_rownames("pathway")

heatmap_data[is.na(heatmap_data)] <- 0

# Build significance-star matrix
sig_data <- sig_pathways %>%
  filter(pathway %in% selected_paths) %>%
  mutate(
    col_label = paste0(stratum, ":", contrast),
    star = case_when(
      padj < 0.001 ~ "***",
      padj < 0.01  ~ "**",
      padj < 0.05  ~ "*",
      TRUE         ~ ""
    )
  ) %>%
  reshape2::dcast(pathway ~ col_label, value.var = "star", fill = "") %>%
  tibble::column_to_rownames("pathway")

# Align columns
sig_data <- sig_data[rownames(heatmap_data), colnames(heatmap_data)]
sig_data[is.na(sig_data)] <- ""

# Clean pathway names for display
clean_name <- function(x) {
  x %>%
    gsub("^(REACTOME_|KEGG_|HALLMARK_|GO_|WP_)", "", .) %>%
    gsub("_", " ", .) %>%
    substr(1, 60)
}
rownames(heatmap_data) <- clean_name(rownames(heatmap_data))
rownames(sig_data)     <- clean_name(rownames(sig_data))

# Draw heatmap
n_rows <- nrow(heatmap_data)
n_cols <- ncol(heatmap_data)
options(repr.plot.width  = max(12, n_cols * 0.6 + 6),
        repr.plot.height = max(8,  n_rows * 0.25 + 2))

pheatmap(
  heatmap_data,
  cluster_rows = TRUE,
  cluster_cols = FALSE,
  breaks = seq(-2, 2, length.out = 101),
  color = colorRampPalette(rev(brewer.pal(9, "RdBu")))(100),
  cellwidth = 18, cellheight = 14,
  border_color = "white",
  display_numbers = as.matrix(sig_data),
  number_color = "black",
  fontsize_number = 7,
  fontsize_row = 8,
  fontsize_col = 8,
  angle_col = 45,
  main = "fGSEA (RUV): Top variable significant pathways"
)

## Section 5 — Diagonal Heatmap: Contrast × Cell-Type Specificity

Permute rows so that pathways cluster along the diagonal, revealing which
pathways are specific to which contrast × cell-type combination.

**Method:** For each pathway (row), find which column has the maximum absolute NES.
Then sort rows by that column index. This produces a "diagonal" pattern where
pathway blocks align with their dominant contrast×celltype.

In [ ]:
# Use all significantly enriched pathways (padj < 0.05, |NES| >= 1)
diag_paths <- gsea %>%
  filter(padj < 0.05, abs(NES) >= 1) %>%
  group_by(stratum, contrast) %>%
  slice_max(order_by = abs(NES), n = 10) %>%
  ungroup() %>%
  pull(pathway) %>% unique()

rmat <- gsea %>%
  filter(pathway %in% diag_paths) %>%
  mutate(col_label = paste0(stratum, ":", contrast)) %>%
  reshape2::dcast(pathway ~ col_label, value.var = "NES") %>%
  tibble::column_to_rownames("pathway")

rmat[is.na(rmat)] <- 0

# --- Diagonal sort: order rows by the column with max |NES| ---
abs_rmat <- abs(as.matrix(rmat))
max_indices <- max.col(abs_rmat, ties.method = "first")
permutation_vector <- order(max_indices)
smat <- rmat[permutation_vector, ]

# Clean pathway names
rownames(smat) <- clean_name(rownames(smat))

n_rows <- nrow(smat)
n_cols <- ncol(smat)
options(repr.plot.width  = max(14, n_cols * 0.55 + 8),
        repr.plot.height = max(10, n_rows * 0.22 + 3))

pheatmap(
  smat,
  cluster_rows = FALSE,   # keep diagonal order
  cluster_cols = FALSE,
  breaks = seq(-2, 2, length.out = 101),
  color = colorRampPalette(rev(brewer.pal(9, "RdBu")))(100),
  cellwidth = 16, cellheight = 12,
  border_color = "white",
  fontsize_row = 7,
  fontsize_col = 8,
  angle_col = 45,
  main = "fGSEA (RUV): Diagonal — pathway specificity by contrast × cell type"
)

## Section 6 — FGSEA Heatmap: Cell-Type-Divergent Pathways

Find pathways that are enriched in **opposite directions** between cell types
for the same contrast (e.g., up in Beta, down in Alpha for Diabetic_vs_ND).
This reveals cell-type-specific biology.

In [ ]:
# For each contrast, find pathways with divergent NES across cell types
divergent_paths <- gsea %>%
  filter(padj < 0.05) %>%
  group_by(pathway, contrast) %>%
  filter(n() >= 2) %>%  # pathway must appear in ≥2 cell types
  summarise(
    has_pos = any(NES > 0),
    has_neg = any(NES < 0),
    nes_sd  = sd(NES),
    .groups = "drop"
  ) %>%
  filter(has_pos & has_neg, nes_sd >= 0.75) %>%
  pull(pathway) %>% unique()

cat(sprintf("%d pathways show divergent enrichment across cell types\n", length(divergent_paths)))

if (length(divergent_paths) >= 3) {
  div_mat <- gsea %>%
    filter(pathway %in% divergent_paths) %>%
    mutate(col_label = paste0(stratum, ":", contrast)) %>%
    reshape2::dcast(pathway ~ col_label, value.var = "NES") %>%
    tibble::column_to_rownames("pathway")

  div_mat[is.na(div_mat)] <- 0

  # Build padj star matrix
  div_stars <- gsea %>%
    filter(pathway %in% divergent_paths) %>%
    mutate(
      col_label = paste0(stratum, ":", contrast),
      star = case_when(
        padj < 0.001 ~ "***",
        padj < 0.01  ~ "**",
        padj < 0.05  ~ "*",
        TRUE         ~ ""
      )
    ) %>%
    reshape2::dcast(pathway ~ col_label, value.var = "star", fill = "") %>%
    tibble::column_to_rownames("pathway")

  div_stars <- div_stars[rownames(div_mat), colnames(div_mat)]
  div_stars[is.na(div_stars)] <- ""

  rownames(div_mat)   <- clean_name(rownames(div_mat))
  rownames(div_stars) <- clean_name(rownames(div_stars))

  n_rows <- nrow(div_mat)
  n_cols <- ncol(div_mat)
  options(repr.plot.width  = max(12, n_cols * 0.6 + 6),
          repr.plot.height = max(8,  n_rows * 0.25 + 2))

  pheatmap(
    div_mat,
    cluster_rows = TRUE,
    cluster_cols = TRUE,
    breaks = seq(-2, 2, length.out = 101),
    color = colorRampPalette(rev(brewer.pal(9, "RdBu")))(100),
    cellwidth = 18, cellheight = 14,
    border_color = "white",
    display_numbers = as.matrix(div_stars),
    number_color = "black",
    fontsize_number = 7,
    fontsize_row = 8,
    fontsize_col = 8,
    angle_col = 45,
    main = "Cell-type-divergent pathways (opposite NES across cell types)"
  )
} else {
  cat("Not enough divergent pathways found. Run more contrasts to populate this plot.\n")
}

## Section 7 — Pipeline Status Overview

In [ ]:
# Status tile plot: contrast × cell type with fine-grained status colors
status_levels <- c("success", "success_base_only",
                   "skipped_no_samples", "skipped_min_group",
                   "skipped_no_pairs", "skipped_preflight", "skipped",
                   "error", "not_run")

status_colors <- c(
  "success"            = "#2ca02c",   # green
  "success_base_only"  = "#98df8a",   # light green
  "skipped_no_samples" = "#ff7f0e",   # orange
  "skipped_min_group"  = "#ffbb78",   # light orange
  "skipped_no_pairs"   = "#f0b27a",   # peach
  "skipped_preflight"  = "#e59866",   # tan
  "skipped"            = "#fad7a0",   # pale orange
  "error"              = "#d62728",   # red
  "not_run"            = "grey80"     # grey
)

status_df <- deseq_all %>%
  mutate(
    status = factor(status, levels = intersect(status_levels, unique(status))),
    contrast = factor(contrast, levels = rev(sort(unique(contrast)))),
    biosample = factor(biosample, levels = sort(unique(biosample))),
    # Best DEG count to display in cells
    deg_label = ifelse(!is.na(n_degs_ruv), n_degs_ruv,
                       ifelse(!is.na(n_degs_base), n_degs_base, ""))
  )

used_colors <- status_colors[names(status_colors) %in% levels(status_df$status)]

options(repr.plot.width = 14, repr.plot.height = 8)

ggplot(status_df, aes(x = biosample, y = contrast, fill = status)) +
  geom_tile(color = "white", linewidth = 0.5) +
  geom_text(aes(label = deg_label), size = 2.2, color = "black") +
  scale_fill_manual(values = used_colors, name = "Status", drop = FALSE) +
  labs(
    x = "Cell type", y = "Contrast", fill = "Status",
    title = "Pipeline completion status (numbers = DEGs from best model)"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    panel.grid = element_blank(),
    legend.position = "bottom"
  ) +
  guides(fill = guide_legend(nrow = 2))

## Section 8 — Export Final DESeq2 Results

Save per-stratum results to `OUT_DIR` with naming convention:
`{stratum}--{contrast}--{model}.{RESULTS_SUFFIX}.tsv`

Each file contains the full DESeq2 results table with `model` and `formula` columns.

In [ ]:
# --- Export final DESeq2 results per stratum ---
if (!nzchar(OUT_DIR)) {
  cat("OUT_DIR is not set. Set it in Section 0 to enable export.\n")
} else if (nrow(deseq_final) == 0) {
  cat("No final DESeq2 results loaded. Run the pipeline first.\n")
} else {
  dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

  # Split by stratum × contrast and write individual files
  export_groups <- deseq_final %>%
    group_by(stratum, contrast, model) %>%
    group_split()

  n_written <- 0
  for (grp in export_groups) {
    s <- unique(grp$stratum)
    c_id <- unique(grp$contrast)
    mdl <- tolower(unique(grp$model))

    fname <- sprintf("%s--%s--%s.%s.tsv", s, c_id, mdl, RESULTS_SUFFIX)
    out_path <- file.path(OUT_DIR, fname)

    fwrite(grp, out_path, sep = "\t")
    n_written <- n_written + 1
  }

  cat(sprintf("Exported %d files to: %s\n", n_written, OUT_DIR))
  cat(sprintf("Naming: {stratum}--{contrast}--{model}.%s.tsv\n", RESULTS_SUFFIX))
}

## Section 9 — Gene Symbol to Ensembl ID Mapping

Map gene symbols in the final results to Ensembl gene IDs using `biomaRt`.
Produces per-stratum files with columns:
`gene | ensembl_id | log2FoldChange | lfcSE | pvalue | padj`

In [ ]:
# --- Build gene symbol → Ensembl ID mapping ---
if (!requireNamespace("biomaRt", quietly = TRUE)) {
  cat("Installing biomaRt...\n")
  if (!requireNamespace("BiocManager", quietly = TRUE)) install.packages("BiocManager")
  BiocManager::install("biomaRt", ask = FALSE, update = FALSE)
}
library(biomaRt)

# Query Ensembl for human gene symbol → Ensembl ID
cat("Connecting to Ensembl BioMart (this may take a moment)...\n")
ensembl <- useMart("ensembl", dataset = "hsapiens_gene_ensembl")

# Get all unique gene symbols from our results
all_genes <- unique(deseq_final$gene)
cat(sprintf("Mapping %d unique gene symbols to Ensembl IDs...\n", length(all_genes)))

gene_map <- getBM(
  attributes = c("hgnc_symbol", "ensembl_gene_id"),
  filters    = "hgnc_symbol",
  values     = all_genes,
  mart       = ensembl
)

# De-duplicate: keep one Ensembl ID per gene symbol (prefer the canonical one)
gene_map <- gene_map %>%
  filter(hgnc_symbol != "", ensembl_gene_id != "") %>%
  group_by(hgnc_symbol) %>%
  slice(1) %>%
  ungroup()

cat(sprintf("Mapped %d / %d genes (%.1f%%)\n",
    nrow(gene_map), length(all_genes),
    100 * nrow(gene_map) / length(all_genes)))

# Preview
head(gene_map, 10)

In [ ]:
# --- Export per-stratum files with Ensembl IDs ---
# Columns: gene, ensembl_id, log2FoldChange, lfcSE, pvalue, padj
if (!nzchar(OUT_DIR)) {
  cat("OUT_DIR is not set. Set it in Section 0 to enable export.\n")
} else {
  ensembl_dir <- file.path(OUT_DIR, "with_ensembl")
  dir.create(ensembl_dir, recursive = TRUE, showWarnings = FALSE)

  # Join Ensembl IDs to the final results
  deseq_with_ensembl <- deseq_final %>%
    left_join(gene_map, by = c("gene" = "hgnc_symbol")) %>%
    rename(ensembl_id = ensembl_gene_id)

  export_groups <- deseq_with_ensembl %>%
    group_by(stratum, contrast, model) %>%
    group_split()

  n_written <- 0
  for (grp in export_groups) {
    s <- unique(grp$stratum)
    c_id <- unique(grp$contrast)
    mdl <- tolower(unique(grp$model))

    # Select the requested columns
    out_df <- grp %>%
      dplyr::select(gene, ensembl_id, log2FoldChange, lfcSE, pvalue, padj) %>%
      arrange(pvalue)

    fname <- sprintf("%s--%s--%s.%s.tsv", s, c_id, mdl, RESULTS_SUFFIX)
    out_path <- file.path(ensembl_dir, fname)
    fwrite(out_df, out_path, sep = "\t")
    n_written <- n_written + 1
  }

  n_unmapped <- sum(is.na(deseq_with_ensembl$ensembl_id))
  cat(sprintf("Exported %d files to: %s\n", n_written, ensembl_dir))
  cat(sprintf("Genes without Ensembl ID: %d (%.1f%% — will have NA in ensembl_id column)\n",
      n_unmapped, 100 * n_unmapped / nrow(deseq_with_ensembl)))
}

## Section 10 — PankBase Submission Manifest

Generate a manifest table for all successful contrast × stratum combinations,
formatted for PankBase submission. This pre-fills all columns except S3 bucket
paths and MD5 checksums (which need to be computed after upload).

The manifest covers:
- `Tabular File_aliases` — the DESeq2 results file
- `Matrix File_aliases` — the RUV-normalized counts file (if applicable)

In [ ]:
# --- PankBase submission manifest ---
# Base paths for NFS and S3 (adjust to your environment)
NFS_RESULTS_BASE <- "/nfs/lab/lbrusman/pankbase/DE_results_EasyDE/final_results"
NFS_RUVNORM_BASE <- "/nfs/lab/lbrusman/pankbase/DE_results_EasyDE/RUVnorm_counts"
S3_BUCKET_BASE   <- "https://pankbase-data-v1.s3.us-west-2.amazonaws.com/analysis_results_files/phenotype_differential_expression"

# Build description text from contrast names
make_description <- function(contrast, biosample) {
  # Convert contrast_id to readable phenotype description
  desc_map <- c(
    "Age"                    = "age",
    "Gender"                 = "sex/gender",
    "BMI"                    = "BMI",
    "Diabetic_vs_ND"         = "diabetic vs non-diabetic status",
    "Prediabetic_vs_ND"      = "prediabetic vs non-diabetic status",
    "Prediabetic_vs_Diabetic" = "prediabetic vs diabetic status",
    "ND_vs_T2D"              = "non-diabetic vs type 2 diabetes status",
    "ND_vs_T1D"              = "non-diabetic vs type 1 diabetes status",
    "T1D_vs_T2D"             = "type 1 vs type 2 diabetes status",
    "HbA1c"                  = "HbA1c levels",
    "C_peptide"              = "C-peptide levels",
    "AAB_GADA"               = "GADA autoantibody presence",
    "AAB_GADA_Pos_Neg"       = "GADA autoantibody positive vs negative",
    "AAB_ZNT8"               = "ZNT8 autoantibody presence",
    "AAB_ZNT8_Pos_Neg"       = "ZNT8 autoantibody positive vs negative",
    "AAB_IAA"                = "IAA autoantibody presence",
    "AAB_IAA_Pos_Neg"        = "IAA autoantibody positive vs negative",
    "AAB_IA2"                = "IA2 autoantibody presence",
    "AAB_IA2_Pos_Neg"        = "IA2 autoantibody positive vs negative",
    "Multi_AAB"              = "multiple autoantibody presence"
  )
  phenotype_desc <- ifelse(contrast %in% names(desc_map),
                           desc_map[contrast], contrast)
  # Clean biosample name for description
  bio_clean <- gsub("_", " ", biosample)
  bio_clean <- gsub("([a-z])([A-Z])", "\\1 \\2", bio_clean)
  bio_clean <- tolower(bio_clean)
  sprintf("Differential expression results testing association with %s in pancreatic %s cells from scRNA-seq",
          phenotype_desc, bio_clean)
}

# Build manifest from successful strata
successful <- deseq_all %>%
  filter(grepl("^success", status))

manifest <- successful %>%
  rowwise() %>%
  mutate(
    Status           = "Released",
    Order            = 1,
    PanKbase_aliases = sprintf("pankbase-consortium:PA_DE_phenotype_%s_%s", biosample, contrast),
    phenotype        = sprintf("%s_%s", biosample, contrast),
    Description      = make_description(contrast, biosample),

    # DESeq2 results file
    results_fname = sprintf("%s_%s_results_final_RUVseq.tsv", biosample, contrast),
    nfs_results   = file.path(NFS_RESULTS_BASE, results_fname),
    `Tabular File_aliases` = sprintf("pankbase-consortium:PA_DE_phenotype_%s_%s_tsv",
                                      biosample, contrast),
    `S3 bucket_final_RUVseq` = file.path(S3_BUCKET_BASE, results_fname),
    `md5_final_RUVseq` = "",   # to be filled after upload

    # RUV-normalized counts file
    ruvnorm_fname = sprintf("%s_%s_RUV_norm_cts.tsv", biosample, contrast),
    nfs_ruvnorm   = file.path(NFS_RUVNORM_BASE, ruvnorm_fname),
    `Matrix File_aliases` = sprintf("pankbase-consortium:PA_DE_phenotype_%s_%s_RUV-normalizedCounts",
                                     biosample, contrast),
    `S3 bucket_RUV_norm_cts` = file.path(S3_BUCKET_BASE, "RUVnorm", ruvnorm_fname),
    `md5_RUV_norm_cts` = ""    # to be filled after upload
  ) %>%
  ungroup() %>%
  dplyr::select(
    Status, Order, PanKbase_aliases, phenotype, Description,
    nfs_results, `Tabular File_aliases`, `S3 bucket_final_RUVseq`, md5_final_RUVseq,
    nfs_ruvnorm, `Matrix File_aliases`, `S3 bucket_RUV_norm_cts`, md5_RUV_norm_cts
  )

cat(sprintf("Manifest: %d rows (%d success, %d success_base_only)\n",
    nrow(manifest),
    sum(successful$status == "success"),
    sum(successful$status == "success_base_only")))

# Preview first rows
head(manifest, 5)

In [ ]:
# --- Save manifest to TSV ---
if (nzchar(OUT_DIR)) {
  manifest_path <- file.path(OUT_DIR, "pankbase_submission_manifest.tsv")
  fwrite(manifest, manifest_path, sep = "\t")
  cat(sprintf("Manifest saved: %s (%d rows)\n", manifest_path, nrow(manifest)))
} else {
  cat("Set OUT_DIR in Section 0 to save the manifest.\n")
}

# --- Also print full table for inspection ---
cat(sprintf("\n=== Full manifest: %d strata x contrast combinations ===\n\n", nrow(manifest)))
print(manifest %>% dplyr::select(phenotype, Description, Status) %>% as.data.frame(), right = FALSE)